# Import Models 

In [1]:
import os
import subprocess
import pandas as pd
import numpy as np
import optuna
from optuna.samplers import NSGAIIISampler
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
import csv
import sys
import time
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

C:\Users\hp\anaconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Build the python script for simulation running 

In [2]:
ECLRUN_EXE = r"C:\ecl\macros\eclrun.exe"

BASE_DIR = r"C:\Users\hp\Desktop\Eclipse 1 optimisation approach"  # we did the simulation for TPE & NSGA on the same DATA file 

DATA_FILE_NAME = "Egg_Model_ECL.DATA"
SCHEDULE_FILE_NAME = "SCHEDULE_NEW.INC"
RSM_PATH = r"C:\Users\hp\Desktop\Eclipse 1 optimisation approach\EGG_MODEL_ECL.RSM"

In [3]:
def modify_files(params):
#____________________________________________________________
    sched_path = os.path.join(BASE_DIR, SCHEDULE_FILE_NAME)

    inj_map = {
        "INJECT1": params["inj_rate1"],
        "INJECT2": params["inj_rate2"],
        "INJECT3": params["inj_rate3"],
        "INJECT4": params["inj_rate4"],
        "INJECT5": params["inj_rate5"],
        "INJECT6": params["inj_rate6"],
        "INJECT7": params["inj_rate7"],
        "INJECT8": params["inj_rate8"],
    }

    with open(sched_path, "r") as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        stripped = line.strip()

        for well, rate in inj_map.items():
            if stripped.startswith(f"'{well}'") and "'RATE'" in stripped:
                parts = stripped.split()

                # Find RATE keyword index safely
                rate_idx = parts.index("'RATE'") + 1
                parts[rate_idx] = str(int(rate))

                line = " ".join(parts) + "\n"

        new_lines.append(line)

    with open(sched_path, "w") as f:
        f.writelines(new_lines)

#_____________________________________________________________
    data_path = os.path.join(BASE_DIR, DATA_FILE_NAME)

    prod_map = {
        "PROD1": params["BHP1"],
        "PROD2": params["BHP2"],
        "PROD3": params["BHP3"],
        "PROD4": params["BHP4"],
    }
    with open(data_path, "r") as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        stripped = line.strip()

        for well, bhp in prod_map.items():
            if stripped.startswith(f"'{well}'") and "'BHP'" in stripped:
                parts = stripped.split()
                parts[-1] = f"{int(bhp)}/"
                line = " ".join(parts) + "\n"

        new_lines.append(line)

    with open(data_path, "w") as f:
        f.writelines(new_lines)

In [4]:
def run_eclipse():
    cmd = [
        ECLRUN_EXE,
        "eclipse",
        DATA_FILE_NAME
    ]
    subprocess.run(cmd, cwd=BASE_DIR, check=True)
    
def extract_FOPT_FWPT_from_RSM(rsm_path):

    with open(rsm_path, "r") as f:
        lines = f.readlines()

    inside_summary = False
    fopt_idx = fwpt_idx = None

    for line in lines:

        if "SUMMARY OF RUN" in line:
            inside_summary = True
            fopt_idx = fwpt_idx = None
            continue

        if not inside_summary:
            continue

        tokens = line.split()
        if not tokens:
            continue

        # Detect column headers
        if fopt_idx is None and "FOPT" in tokens and "FWPT" in tokens:
            fopt_idx = tokens.index("FOPT")
            fwpt_idx = tokens.index("FWPT")
            continue

        if fopt_idx is None:
            continue

        # ✅ Only data rows have numeric TIME
        if not tokens[0][0].isdigit():
            continue
        if float(tokens[0]) == 3600.0:
            return float(tokens[fopt_idx]), float(tokens[fwpt_idx])

# Define the TPE objective function

In [9]:
# ------------------------------------------------------------------
# Settings
# ------------------------------------------------------------------
SEEDS = [42, 123, 2026]       # 3 seeds to test stability (change freely)
N_TRIALS = 300
N_STARTUP = 42                # random start-up trials; TPE takes over after this
MULTIVARIATE = False
STUDY_BASE = "ECLIPSE_SUM_DELTA_FOPT_FWPT"
STORAGE = "sqlite:///egg_tpe_mo_seed.db"   # all seeds saved in one file

PARAM_BOUNDS = {
    **{f"BHP{i}": (150, 390) for i in range(1, 5)},
    **{f"inj_rate{i}": (50, 400) for i in range(1, 9)},
}

C_RANDOM, C_TPE, C_PARETO = "#a3a29b", "#2b7bd6", "#ee6a33"
SEED_COLORS = ["#2b7bd6", "#ee6a33", "#1baf7a", "#8e5bd6", "#d6a21b"]
SEED_MARKERS = ["o", "s", "^", "D", "v"]

plt.rcParams.update({
    "axes.facecolor": "#fafaf8", "axes.edgecolor": "#bbbbbb",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e6e6", "font.size": 13,
})


def study_name(seed):
    return f"{STUDY_BASE}_seed{seed}"


# ------------------------------------------------------------------
# Objective: timings are stored on the trial itself (user_attrs)
# ------------------------------------------------------------------
def objective(trial):
    # 1. Sampling (random during start-up, TPE afterwards)
    t0 = time.perf_counter()
    params = {name: trial.suggest_int(name, lo, hi)
              for name, (lo, hi) in PARAM_BOUNDS.items()}
    trial.set_user_attr("t_sampling", time.perf_counter() - t0)

    # 2. Write Eclipse input files
    modify_files(params)

    # 3. Eclipse simulation (time recorded even if the run crashes)
    t0 = time.perf_counter()
    try:
        run_eclipse()
    finally:
        trial.set_user_attr("t_eclipse", time.perf_counter() - t0)

    # 4. Objectives
    fopt, fwpt = extract_FOPT_FWPT_from_RSM(RSM_PATH)
    return fopt, fwpt


# ------------------------------------------------------------------
# Run one seed (resumes an interrupted study instead of starting over)
# ------------------------------------------------------------------
def run_seed(seed):
    sampler = optuna.samplers.TPESampler(
        n_startup_trials=N_STARTUP, multivariate=MULTIVARIATE, seed=seed)
    study = optuna.create_study(
        directions=["maximize", "minimize"], sampler=sampler,
        study_name=study_name(seed), storage=STORAGE, load_if_exists=True)
    study.set_user_attr("seed", seed)
    study.set_user_attr("n_startup", N_STARTUP)

    done = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)
    remaining = N_TRIALS - done
    print(f"[seed {seed}] {done} trials already done, running {max(remaining, 0)} more")
    if remaining > 0:
        # catch=...: a crashed Eclipse run is logged as FAIL instead of stopping the study
        study.optimize(objective, n_trials=remaining, show_progress_bar=False,
                       catch=(Exception,))
    return study


# ------------------------------------------------------------------
# Figure 1 (per seed): sampled values
# ------------------------------------------------------------------
def plot_sampled_values(study, n_startup, fname, window=30):
    trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    pareto = {t.number for t in study.best_trials}
    num = np.array([t.number for t in trials])
    is_rnd = num < n_startup
    is_par = np.isin(num, list(pareto))
    seed = study.user_attrs.get("seed", "?")

    fig, axes = plt.subplots(3, 4, figsize=(24, 14), sharex=True)
    for ax, name in zip(axes.flat, PARAM_BOUNDS):
        val = np.array([t.params[name] for t in trials], dtype=float)

        # 10-90 % band of the last `window` TPE samples
        tn, tv = num[~is_rnd], val[~is_rnd]
        if len(tv) > 1:
            lo = [np.percentile(tv[max(0, i - window + 1):i + 1], 10) for i in range(len(tv))]
            hi = [np.percentile(tv[max(0, i - window + 1):i + 1], 90) for i in range(len(tv))]
            ax.fill_between(tn, lo, hi, color=C_TPE, alpha=0.12, lw=0, step="post")

        ax.scatter(num[is_rnd], val[is_rnd], s=18, c=C_RANDOM, zorder=2)
        ax.scatter(tn, tv, s=18, c=C_TPE, zorder=3)
        ax.scatter(num[is_par], val[is_par], s=90, facecolors="none",
                   edgecolors=C_PARETO, lw=2, zorder=4)

        b_lo, b_hi = PARAM_BOUNDS[name]
        ax.set_ylim(b_lo, b_hi)
        ax.set_title(name, fontsize=17)
    for ax in axes[-1]:
        ax.set_xlabel("Trial number", fontsize=15)

    handles = [
        Line2D([], [], ls="", marker="o", color=C_RANDOM, label="random start-up"),
        Line2D([], [], ls="", marker="o", color=C_TPE, label="TPE"),
        Line2D([], [], ls="", marker="o", mfc="none", mec=C_PARETO, mew=2, ms=10,
               label="Pareto-optimal"),
    ]
    fig.legend(handles=handles, loc="upper center", ncol=3, frameon=False,
               fontsize=15, bbox_to_anchor=(0.5, 1.0))
    fig.suptitle(f"Seed {seed} - sampled values: random start-up ({n_startup} trials), then "
                 f"concentration of TPE samples (band = 10-90 % of the last {window} TPE samples)",
                 y=0.965, fontsize=17)
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    fig.savefig(fname, dpi=200)
    plt.close(fig)


# ------------------------------------------------------------------
# Figure 2 (per seed): time per trial (Eclipse vs sampling) + totals
# ------------------------------------------------------------------
def plot_timing(study, n_startup, fname):
    trials = [t for t in study.trials
              if "t_eclipse" in t.user_attrs and "t_sampling" in t.user_attrs]
    num = np.array([t.number for t in trials])
    t_sim = np.array([t.user_attrs["t_eclipse"] for t in trials])
    t_smp = np.array([t.user_attrs["t_sampling"] for t in trials])
    is_rnd = num < n_startup
    seed = study.user_attrs.get("seed", "?")

    fig = plt.figure(figsize=(22, 7.5))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.75, 1], wspace=0.04)

    ax = fig.add_subplot(gs[0])
    ax.scatter(num, t_sim, s=22, c=C_TPE, label="Eclipse simulation")
    ax.scatter(num[is_rnd], t_smp[is_rnd], s=22, c=C_RANDOM, label="random start-up sampling")
    ax.scatter(num[~is_rnd], t_smp[~is_rnd], s=22, c=C_PARETO, label="TPE sampling")
    ax.set_yscale("log")
    ax.axvline(n_startup - 0.5, ls="--", color="#999999")
    ax.text(n_startup + 2, 1.0, "TPE starts", transform=ax.get_xaxis_transform(),
            va="bottom", color="#666666")
    ax.set_xlabel("Trial number")
    ax.set_ylabel("Time per trial (s, log scale)")
    ax.set_title(f"Seed {seed} - time per trial: Eclipse vs TPE", pad=28)
    ax.legend(loc="center right", frameon=False)

    h_sim, h_smp = t_sim.sum() / 3600, t_smp.sum() / 3600
    tot = h_sim + h_smp
    ax2 = fig.add_subplot(gs[1])
    ax2.barh([0], [h_sim], color=C_TPE, height=0.5)
    ax2.barh([0], [h_smp], left=[h_sim], color=C_PARETO, height=0.5)
    ax2.set_ylim(-1.6, 0.5)
    ax2.set_yticks([])
    ax2.grid(axis="y", visible=False)
    ax2.set_xlabel("Total time (h)")
    ax2.set_title(f"Total {tot:.2f} h for {len(trials)} trials", pad=28)
    ax2.legend(handles=[
        Patch(color=C_TPE, label=f"Eclipse: {h_sim:.3f} h ({100 * h_sim / tot:.2f} %)"),
        Patch(color=C_PARETO, label=f"Sampling (random + TPE): {h_smp:.4f} h "
                                    f"({100 * h_smp / tot:.3f} %)"),
    ], loc="lower left", frameon=False, fontsize=14)

    fig.subplots_adjust(left=0.06, right=0.98, bottom=0.12, top=0.86)
    fig.savefig(fname, dpi=200)
    plt.close(fig)


# ------------------------------------------------------------------
# Stability metric: 2-D hypervolume, normalised over all seeds' fronts
# ------------------------------------------------------------------
def pareto_values(study):
    """Pareto front as an (n, 2) array [FOPT, FWPT], sorted by FWPT."""
    v = np.array([t.values for t in study.best_trials], dtype=float)
    return v[np.argsort(v[:, 1])]


def hypervolumes(fronts):
    """Hypervolume of each front after min-max normalisation on the union of all fronts.
    Both objectives are turned into 'minimise' and scaled to [0, 1]; reference point = (1.1, 1.1).
    Max possible value is 1.21; similar values across seeds = stable workflow."""
    allv = np.vstack(fronts)
    lo, hi = allv.min(axis=0), allv.max(axis=0)
    span = np.where(hi - lo > 0, hi - lo, 1.0)
    out = []
    for f in fronts:
        g1 = (hi[0] - f[:, 0]) / span[0]          # FOPT (max)  -> minimise
        g2 = (f[:, 1] - lo[1]) / span[1]          # FWPT (min)
        pts = np.column_stack([g1, g2])
        pts = pts[np.lexsort((pts[:, 1], pts[:, 0]))]
        hv, best_g2, ref = 0.0, np.inf, 1.1
        xs = np.append(pts[:, 0], ref)
        for i, (x, y) in enumerate(pts):
            best_g2 = min(best_g2, y)
            hv += (xs[i + 1] - x) * (ref - best_g2)
        out.append(hv)
    return out


# ------------------------------------------------------------------
# Figure 3 (combined): Pareto fronts of all seeds on one plot
# ------------------------------------------------------------------
def plot_pareto_fronts(studies, fname="pareto_fronts_by_seed.png"):
    fronts = [pareto_values(s) for s in studies]
    hvs = hypervolumes(fronts)

    fig, ax = plt.subplots(figsize=(11, 7.5))

    for k, (study, front, hv) in enumerate(zip(studies, fronts, hvs)):
        seed = study.user_attrs.get("seed", "?")
        color = SEED_COLORS[k % len(SEED_COLORS)]
        marker = SEED_MARKERS[k % len(SEED_MARKERS)]

        ax.scatter(
            front[:, 1],
            front[:, 0],
            color=color,
            marker=marker,
            s=65,
            alpha=0.8,
            edgecolors="white",
            linewidths=0.8,
            label=f"Seed {seed} ({len(front)} points, HV = {hv:.3f})"
        )

    ax.set_xlabel("FWPT - field water production total (minimise)")
    ax.set_ylabel("FOPT - field oil production total (maximise)")

    ax.set_title(
        f"Pareto solutions for {len(studies)} TPE seeds "
        f"({N_TRIALS} trials each, {N_STARTUP} random start-up)"
    )

    ax.ticklabel_format(style="sci", axis="both", scilimits=(-3, 4))
    ax.legend(loc="lower right", frameon=False)

    ax.grid(True, linestyle="--", alpha=0.25)

    fig.tight_layout()
    fig.savefig(fname, dpi=300)
    plt.close(fig)

    return fronts, hvs


def write_summary(studies, fronts, hvs, fname="pareto_summary_by_seed.csv"):
    rows = []
    for study, front, hv in zip(studies, fronts, hvs):
        t_sim = sum(t.user_attrs.get("t_eclipse", 0.0) for t in study.trials)
        t_smp = sum(t.user_attrs.get("t_sampling", 0.0) for t in study.trials)
        rows.append({
            "seed": study.user_attrs.get("seed"),
            "n_complete": sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials),
            "n_pareto": len(front),
            "max_FOPT": front[:, 0].max(),
            "min_FWPT": front[:, 1].min(),
            "hypervolume_norm": round(hv, 4),
            "eclipse_h": round(t_sim / 3600, 3),
            "sampling_h": round(t_smp / 3600, 4),
        })
    with open(fname, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)

    hv = np.array(hvs)
    print("\n" + " | ".join(rows[0]))
    for r in rows:
        print(" | ".join(str(v) for v in r.values()))
    print(f"\nHypervolume across seeds: mean {hv.mean():.4f}, std {hv.std(ddof=1):.4f}, "
          f"CV {100 * hv.std(ddof=1) / hv.mean():.2f} %")

# TPE optimization running

In [10]:
# ------------------------------------------------------------------
if __name__ == "__main__":
    studies = []
    for seed in SEEDS:
        if "--plot-only" in sys.argv:
            study = optuna.load_study(study_name=study_name(seed), storage=STORAGE)
        else:
            study = run_seed(seed)
        studies.append(study)

        n_startup = study.user_attrs.get("n_startup", N_STARTUP)
        plot_sampled_values(study, n_startup, f"sampled_values_seed{seed}.png")
        plot_timing(study, n_startup, f"time_per_trial_seed{seed}.png")

    fronts, hvs = plot_pareto_fronts(studies)
    write_summary(studies, fronts, hvs)

[I 2026-11-04 16:57:04,241] A new study created in RDB with name: ECLIPSE_SUM_DELTA_FOPT_FWPT_seed42


[seed 42] 0 trials already done, running 300 more


[I 2026-11-04 16:58:22,798] Trial 0 finished with values: [542456.4, 5351730.0] and parameters: {'BHP1': 240, 'BHP2': 379, 'BHP3': 326, 'BHP4': 294, 'inj_rate1': 104, 'inj_rate2': 104, 'inj_rate3': 70, 'inj_rate4': 354, 'inj_rate5': 260, 'inj_rate6': 298, 'inj_rate7': 57, 'inj_rate8': 390}. 
[I 2026-11-04 16:59:51,752] Trial 1 finished with values: [538358.4, 4629445.0] and parameters: {'BHP1': 350, 'BHP2': 201, 'BHP3': 193, 'BHP4': 194, 'inj_rate1': 156, 'inj_rate2': 234, 'inj_rate3': 201, 'inj_rate4': 152, 'inj_rate5': 264, 'inj_rate6': 98, 'inj_rate7': 152, 'inj_rate8': 178}. 
[I 2026-11-04 17:01:56,074] Trial 2 finished with values: [541525.1, 6195296.0] and parameters: {'BHP1': 259, 'BHP2': 339, 'BHP3': 198, 'BHP4': 273, 'inj_rate1': 257, 'inj_rate2': 66, 'inj_rate3': 263, 'inj_rate4': 109, 'inj_rate5': 72, 'inj_rate6': 383, 'inj_rate7': 388, 'inj_rate8': 333}. 
[I 2026-11-04 17:03:41,202] Trial 3 finished with values: [534377.2, 5079976.0] and parameters: {'BHP1': 223, 'BHP2': 17

[seed 123] 0 trials already done, running 300 more


[I 2026-11-05 01:07:06,386] Trial 0 finished with values: [554435.9, 6877528.0] and parameters: {'BHP1': 317, 'BHP2': 218, 'BHP3': 204, 'BHP4': 282, 'inj_rate1': 302, 'inj_rate2': 198, 'inj_rate3': 394, 'inj_rate4': 290, 'inj_rate5': 218, 'inj_rate6': 187, 'inj_rate7': 170, 'inj_rate8': 305}. 
[I 2026-11-05 01:08:45,122] Trial 1 finished with values: [549345.4, 6238654.0] and parameters: {'BHP1': 255, 'BHP2': 164, 'BHP3': 245, 'BHP4': 327, 'inj_rate1': 114, 'inj_rate2': 111, 'inj_rate3': 236, 'inj_rate4': 236, 'inj_rate5': 272, 'inj_rate6': 348, 'inj_rate7': 304, 'inj_rate8': 264}. 
[I 2026-11-05 01:10:06,603] Trial 2 finished with values: [536551.6, 4829000.0] and parameters: {'BHP1': 324, 'BHP2': 227, 'BHP3': 237, 'BHP4': 205, 'inj_rate1': 153, 'inj_rate2': 271, 'inj_rate3': 82, 'inj_rate4': 202, 'inj_rate5': 201, 'inj_rate6': 223, 'inj_rate7': 199, 'inj_rate8': 159}. 
[I 2026-11-05 01:11:11,070] Trial 3 finished with values: [546293.6, 6006726.0] and parameters: {'BHP1': 252, 'BHP2'

[seed 2026] 0 trials already done, running 300 more


[I 2026-11-05 08:51:02,978] Trial 0 finished with values: [553917.6, 6708978.0] and parameters: {'BHP1': 202, 'BHP2': 249, 'BHP3': 385, 'BHP4': 171, 'inj_rate1': 218, 'inj_rate2': 396, 'inj_rate3': 119, 'inj_rate4': 370, 'inj_rate5': 245, 'inj_rate6': 324, 'inj_rate7': 156, 'inj_rate8': 189}. 
[I 2026-11-05 08:52:18,263] Trial 1 finished with values: [541061.6, 5396946.0] and parameters: {'BHP1': 221, 'BHP2': 365, 'BHP3': 161, 'BHP4': 257, 'inj_rate1': 247, 'inj_rate2': 177, 'inj_rate3': 293, 'inj_rate4': 374, 'inj_rate5': 60, 'inj_rate6': 92, 'inj_rate7': 131, 'inj_rate8': 275}. 
[I 2026-11-05 08:53:19,748] Trial 2 finished with values: [553859.4, 6518353.0] and parameters: {'BHP1': 206, 'BHP2': 383, 'BHP3': 185, 'BHP4': 161, 'inj_rate1': 215, 'inj_rate2': 289, 'inj_rate3': 358, 'inj_rate4': 289, 'inj_rate5': 232, 'inj_rate6': 213, 'inj_rate7': 140, 'inj_rate8': 228}. 
[I 2026-11-05 08:54:50,322] Trial 3 finished with values: [544029.3, 5120033.0] and parameters: {'BHP1': 288, 'BHP2':


seed | n_complete | n_pareto | max_FOPT | min_FWPT | hypervolume_norm | eclipse_h | sampling_h
42 | 300 | 34 | 560304.8 | 2405720.0 | 0.8226 | 8.079 | 0.0505
123 | 300 | 34 | 562767.4 | 2092443.0 | 0.8407 | 7.678 | 0.0503
2026 | 300 | 51 | 559263.4 | 2011719.0 | 0.8552 | 10.771 | 0.0837

Hypervolume across seeds: mean 0.8395, std 0.0163, CV 1.94 %
